# Image inference

Use the PanopTILs model from [fine-tuning](finetuning.ipynb) to find nuclei and
cell types in a held-out tissue region. Each instance label identifies one nucleus;
a class label gives its type. You will compare predictions with the dataset labels
and can save the predicted masks.

Run preparation and fine-tuning first, then open this notebook from `docs/tutorials/`.
It reuses their patient-separated data and final checkpoint. For inference with an
existing public checkpoint on your own image, see the [quick start](../quickstart.md).
A CUDA GPU is preferred, with CPU as the fallback.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Patch
from skimage.color import label2rgb

from cellseg_models_pytorch.inference.post_processor import PostProcessor
from cellseg_models_pytorch.metrics import panoptic_quality
from cellseg_models_pytorch.models.cellpose import CellPose
from cellseg_models_pytorch.transforms.functional.normalization import minmax_normalize
from cellseg_models_pytorch.utils import FileHandler

torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
data_root = Path("../../data/panoptils-nuclei")
checkpoint_path = Path("finetuning-checkpoints/epoch_05.pt")

# Restore the architecture and class meanings together with the learned weights.
state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
class_names = state["class_names"]
model = CellPose(len(class_names), enc_name=state["encoder"], enc_pretrain=False, device=device)
model.model.load_state_dict(state["model"], strict=True)
model.set_inference_mode(mixed_precision=False)
model.post_processor = PostProcessor("cellpose", {"use_gpu": device.type == "cuda"})

# Use the first predetermined validation region at its original 1024-pixel scale.
image_path = sorted((data_root / "validation/regions/images").glob("*.png"))[0]
raw_image = FileHandler.read_img(image_path)
labels = FileHandler.read_mat(data_root / "validation/regions/labels" / f"{image_path.stem}.mat")
image = minmax_normalize(raw_image, copy=True)


In [ ]:
# Dense predictions become individual nuclei and one class label per nucleus.
prediction = model.predict(image)
result = model.post_process(prediction, use_async_postproc=False)
instances, classes = result["nuc"][0]

print("Predicted nuclei:", len(np.unique(instances[instances > 0])))
print("Instance PQ:", round(panoptic_quality(labels["inst"], instances)["pq"], 3))


## Compare the masks

Compare the same tissue image with target and predicted instances and cell types.
Panoptic quality (PQ) checks nucleus matching and overlap; 1 is perfect. This
single image is an illustration; the fine-tuning page checks 16 held-out regions.

Use the fixed class legend to distinguish tumour (neoplastic), supporting tissue
(stromal) and immune (inflammatory) cells. Unknown means the dataset did not assign
a specific type. Brown objects received class 0; black is background with no instance.


In [ ]:
class_palette = np.array([
    [0.55, 0.3, 0.15], [0.9, 0.2, 0.2], [0.2, 0.5, 0.9], [0.2, 0.75, 0.3],
    [0.8, 0.3, 0.8], [0.95, 0.7, 0.15], [0.6, 0.6, 0.6],
])
instance_colors = label2rgb(instances, bg_label=0)
class_colors = class_palette[classes]
class_colors[instances == 0] = 0

target_class_colors = class_palette[labels["type"].astype(int)]
target_class_colors[labels["inst"] == 0] = 0

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(raw_image)
axes[0, 0].set_title("Held-out image")
axes[0, 1].imshow(label2rgb(labels["inst"], bg_label=0))
axes[0, 1].set_title("Target instances")
axes[0, 2].imshow(target_class_colors)
axes[0, 2].set_title("Target cell types")
axes[1, 0].imshow(raw_image)
axes[1, 0].set_title("Held-out image")
axes[1, 1].imshow(instance_colors)
axes[1, 1].set_title("Predicted instances")
axes[1, 2].imshow(class_colors)
axes[1, 2].set_title("Predicted cell types")
for axis in axes.flat:
    axis.set(xticks=[], yticks=[])
fig.legend([Patch(color=colour) for colour in class_palette], class_names,
           loc="lower center", ncol=7)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()


## Save the masks

Choose a new output file, then uncomment the block below. Instance mask zero means
background; each positive number identifies one nucleus. Class IDs retain the
PanopTILs meanings saved in the checkpoint. Saving integer arrays preserves labels.

For larger images, the existing `predict(..., use_sliding_win=True,
window_size=(256, 256), stride=128)` API predicts overlapping tiles. Match image
resolution and preprocessing to your checkpoint rather than shrinking nuclei for speed.


In [ ]:
output_path = Path("segmentation.npz")
# Uncomment after choosing a destination; "xb" protects an existing result.
# with output_path.open("xb") as output:
#     np.savez_compressed(output, instances=instances, classes=classes)
